In [ ]:

import os

os.environ['KAGGLE_USERNAME'] = 'abdulrehman810'
os.environ['KAGGLE_KEY'] = 'KGAT_986dc3e90d9382081886fe5c33c4f57e'


!kaggle datasets download -d meowmeowmeowmeowmeow/gtsrb-german-traffic-sign
!unzip -o gtsrb-german-traffic-sign.zip



In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import cv2
import time
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:
IMG_SIZE = 32

def preprocess(img):
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    # BGR → HSV
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Normalize
    img = img / 255.0

    return img, hsv, gray


prewitt_x = np.array([[-1,0,1],[-1,0,1],[-1,0,1]])
laplacian = np.array([[0,1,0],[1,-4,1],[0,1,0]])

def apply_filter(img, kernel):
    return cv2.filter2D(img, -1, kernel)

# example
img = cv2.imread('/content/Train/0/00000_00000_00000.png')
img, hsv, gray = preprocess(img)

prewitt_img = apply_filter(gray, prewitt_x)
lap_img = apply_filter(gray, laplacian)

sobel_img = cv2.Sobel(gray, cv2.CV_64F, 1, 0)

plt.figure(figsize=(10,3))
plt.subplot(1,4,1); plt.imshow(gray, cmap='gray'); plt.title("Gray")
plt.subplot(1,4,2); plt.imshow(prewitt_img, cmap='gray'); plt.title("Prewitt")
plt.subplot(1,4,3); plt.imshow(lap_img, cmap='gray'); plt.title("Laplacian")
plt.subplot(1,4,4); plt.imshow(sobel_img, cmap='gray'); plt.title("Sobel")
plt.show()

In [ ]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_gen.flow_from_directory(
    '/content/Train',
    target_size=(32,32),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = train_gen.flow_from_directory(
    '/content/Train',
    target_size=(32,32),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)


In [ ]:
model_fc = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(32,32,3)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(43, activation='softmax')
])

model_fc.compile(optimizer='adam',
                 loss='categorical_crossentropy',
                 metrics=['accuracy'])

model_fc.fit(train_data, validation_data=val_data, epochs=2)

In [ ]:
model_cnn = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32,(3,3),activation='relu', input_shape=(32,32,3)),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128,activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(43,activation='softmax')
])

model_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model_cnn.fit(train_data, validation_data=val_data, epochs=2)


In [ ]:
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.show()

In [ ]:
base = tf.keras.applications.ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=(32,32,3)
)

base.trainable = False

x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
out = tf.keras.layers.Dense(43, activation='softmax')(x)

model_resnet = tf.keras.Model(base.input, out)

model_resnet.compile(optimizer='adam',
                     loss='categorical_crossentropy',
                     metrics=['accuracy'])

start = time.time()
model_resnet.fit(train_data, validation_data=val_data, epochs=5)
resnet_time = time.time() - start

In [ ]:
base = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(32,32,3)
)

base.trainable = False

x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
out = tf.keras.layers.Dense(43, activation='softmax')(x)

model_eff = tf.keras.Model(base.input, out)

model_eff.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

start = time.time()
model_eff.fit(train_data, validation_data=val_data, epochs=5)
eff_time = time.time() - start


In [ ]:
resnet_acc = model_resnet.evaluate(val_data)[1]
eff_acc = model_eff.evaluate(val_data)[1]

models = ['ResNet50', 'EfficientNetB0']
acc = [resnet_acc, eff_acc]
times = [resnet_time, eff_time]

plt.bar(models, acc)
plt.title("Accuracy Comparison")
plt.show()

plt.bar(models, times)
plt.title("Inference Time Comparison")
plt.show()